# Geometry-only collapse clustering

This is an unsupervised exploration. RAPID timing response is excluded when fitting clusters and is used only afterward to interpret whether geometric groups correspond to distinct response patterns.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'RAPID').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_IDS = ['02', '03', '05', '11','13','15', '21', '56']  # edit when more real-network outputs are available
RUN_DIR = REPO_ROOT / 'RAPID' / 'classification' / 'results' / f"clustering-notebook-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"


In [ ]:
command = [sys.executable, str(REPO_ROOT / 'RAPID/classification/train_rapid_geometry_clustering.py'),
           '--output-root', str(OUTPUT_ROOT), '--results-dir', str(RUN_DIR),
           '--example-ids', *EXAMPLE_IDS]
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
summary = pd.read_csv(RUN_DIR / 'clustering_quality_summary.csv')
display(summary)
valid = summary.loc[summary.valid_exploratory_solution].dropna(subset=['silhouette'])
best = valid.iloc[0]['algorithm']
print('Highest-silhouette supported candidate:', best)


In [ ]:
assignments = pd.read_csv(RUN_DIR / 'cluster_assignments.csv')
plot_df = assignments.loc[assignments.algorithm.eq(best)].copy()
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(plot_df.len_fraction_width_weighted_log_lambda, plot_df.along_network_frac, c=plot_df.cluster, cmap='tab10', alpha=.8)
ax.set(xlabel='Length- and width-weighted −log10(Lambda)', ylabel='Along-network fraction', title=f'{best}: geometry-only clusters')
fig.colorbar(scatter, ax=ax, label='Cluster')
plt.show()
display(plot_df.groupby('cluster').agg(n=('state_id','size'), response_mean=('lag_to_inflow_peak_pct_shift_vs_base','mean'), response_median=('lag_to_inflow_peak_pct_shift_vs_base','median'), examples=('example_id','nunique')))
